# SI — effect of pixel width on localisation and spectral-fingerprint precision

**Makes up:** SI `Limitations/Pixel_Size_StandardUnits`: a) localisation precision σ_xy and
b) spectral-fingerprint precision σ_colour against pixel width, for ATTO 488, ATTO 565 and ATTO 647N
emitting 4,000 photons. The dashed line is the Ximea's 69 nm.

**Simulation:** the paper's simulator (`MultiC_Sim_Funcs.test_simulation_method`): each photon is
given a wavelength from the dye's emission spectrum, lands at a position drawn from the vectorial
dipole PSF at that wavelength (NA 1.49), and is detected with its pixel's QE at that wavelength;
the Ximea's Bayer pattern (BGGR) and its median gain, offset, read noise and relative QE; no
background. Each simulated molecule is fitted with `STANDARD_DATA` and the Δχ² fit gate, as the
real data, with a starting background suited to large ROIs (section 1b).
Only the pixel width changes; the image is kept at least 1.5 µm across (and at least 7
Bayer unit cells), so the PSF is never cut off.

**Metrics** (as the SBR figures, S34–S36): σ_xy = √((var(x error) + var(y error)) / 2), in nm;
σ_colour = standard deviation of the distance between the fitted (A_B, A_G, A_R) and the dye's
true fingerprint, in units of relative pixel QE.

## FAST_MODE

`FAST_MODE = True` (default): pixel widths 30–200 nm in 10 nm steps, 300 molecules each (a few
minutes; noisy). `FAST_MODE = False`: 30–200 nm in 5 nm steps, 10,000 molecules each (the
published figure used 50,000; hours, run on a workstation).

---
## 0 — Imports

In [ ]:
import logging
logging.basicConfig(level=logging.WARNING, format='%(name)s — %(levelname)s — %(message)s')

import glob
import time
import types
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import xarray as xr
from scipy.spatial.distance import cdist

from pyS3M import IOFunctions, SpectralFunctions, MaskFunctions, sCMOSFunctions, PlottingBase
from pyS3M.simulation.multicolour import FittingStrategy, SimulationConfig, MultiC_Sim_Funcs

IO      = IOFunctions.IO_Functions()
S_F     = SpectralFunctions.Spectral_Funcs()
M_F     = MaskFunctions.Mask_Functions()
sCMOS   = sCMOSFunctions.sCMOS_Functions()
MSF     = MultiC_Sim_Funcs()
plotter = PlottingBase.PublicationPlotter()

smoothing_function = types.SimpleNamespace(
    args={'sigma': 1.5}, extent=1.5, smoothing_function=sCMOS.gaussian_filter_stack, data_arg='image',
)

---
## 1 — Configuration

In [ ]:
FAST_MODE = True   # False -> 30-200 nm in 5 nm steps, 10,000 molecules per point
OVERWRITE = False  # True: simulate again even if the results exist

REPO_ROOT = Path.cwd().resolve().parents[2]   # notebooks/figures/SI -> repo root
CALIBRATION_DIR = REPO_ROOT / "Camera_Calibrations" / "Ximea_Camera"
OUT_ROOT  = Path("outputs") / "FigureSI_PixelSize"   # git-ignored (bare outputs/ rule)
FIG_DIR   = OUT_ROOT / "figures"
SIM_DIR   = OUT_ROOT / ("sim_fast" if FAST_MODE else "sim_full")
FIG_DIR.mkdir(parents=True, exist_ok=True)
SIM_DIR.mkdir(parents=True, exist_ok=True)

DYES = ['ATTO 488', 'ATTO 565', 'ATTO 647N']
COLOURS = ['#a46cb7', '#7aa457', '#cb6a49']
N_PHOTONS = 4000
NA = 1.49
XIMEA_PIXEL_NM = 69.0
MIN_FOV_NM = 1500.0   # simulated image at least this wide

PIXEL_SIZES = np.arange(30, 201, 10 if FAST_MODE else 5).astype(float)
N_BOOTSTRAP = 300 if FAST_MODE else 10_000

gain      = IO.read_tiff(str(CALIBRATION_DIR / 'gain.tif'))
offset    = IO.read_tiff(str(CALIBRATION_DIR / 'offset.tif'))
variance  = IO.read_tiff(str(CALIBRATION_DIR / 'variance.tif'))
readnoise = float(np.median(IO.read_tiff(str(CALIBRATION_DIR / 'readnoise.tif'))))
rqe       = IO.read_tiff(str(CALIBRATION_DIR / 'rqe.tif'))

R_base, G_base, B_base, wavelength = S_F.getpixelefficiency()
pixel_QYs = np.vstack([B_base, G_base, R_base])   # B=0, G=1, R=2
mosaic = M_F.mosaic_unit                           # Ximea BGGR
image_size = 14                                    # seed size; resized to n_unit_cells x mosaic

camera_params = {
    'gain':                np.full((image_size, image_size), np.median(gain)),
    'offset':              np.full((image_size, image_size), np.median(offset)),
    'variance':            np.full((image_size, image_size), np.median(variance)),
    'readnoise':           readnoise,
    'rqe':                 np.full((image_size, image_size), np.median(rqe)),
    'masks':               M_F.get_masks(size_x=image_size, size_y=image_size),
    'pixel_QYs':           pixel_QYs,
    'pixel_order':         ['B', 'G', 'R'],
    'pixel_order_indices': {'B': 0, 'G': 1, 'R': 2},
    'mosaic_unit':         mosaic,
}


def n_unit_cells(pixel_nm):
    return max(7, int(np.ceil(MIN_FOV_NM / (pixel_nm * mosaic.shape[0]))))


def flag(pixel_nm):
    return f"px{pixel_nm:05.1f}nm_".replace('.', 'p')


print(f"FAST_MODE={FAST_MODE}: {len(PIXEL_SIZES)} pixel widths ({PIXEL_SIZES[0]:.0f}-{PIXEL_SIZES[-1]:.0f} nm), "
      f"{N_BOOTSTRAP:,} molecules each, {len(DYES)} dyes")
print(f"Image: {n_unit_cells(PIXEL_SIZES[0]) * 2} px at {PIXEL_SIZES[0]:.0f} nm ... "
      f"{n_unit_cells(PIXEL_SIZES[-1]) * 2} px at {PIXEL_SIZES[-1]:.0f} nm; read noise {readnoise:.2f} e-")
print(f"Results: {SIM_DIR.resolve()}\nFigures: {FIG_DIR.resolve()}")

---
## 1b — Fitter starting point for wide PSFs (this notebook only)

The fitter's starting background for each Bayer channel is the smallest |photoelectron value| in
that channel (at least 1 pe), which suits the 12-pixel ROIs of the real data. Here the whole image
is the ROI, and narrow pixels make it large (50 × 50 px at 30 nm, so the PSF is ~4 px wide): with no
background, that guess (~0.8 pe per pixel, from the quantised photoelectron values) times 625 red
pixels exceeds the molecule's red signal. The starting amplitude then hits its floor, the first
Levenberg–Marquardt step throws the PSF out of the image, and almost every fit fails below ~40 nm.

So for this notebook only, each channel's starting background is the mean of its pixels in the
image's outer ring (width 1/8 of the image, at least 2 px; well clear of the PSF), at least 0.1 pe,
and the starting amplitudes are recomputed from it; the position and width guesses are unchanged.
(The mean, not the median: photoelectron values are quantised, so the median can sit a whole level,
~0.8 pe, above zero.) Checked on 60–200 molecules per point: ATTO 488, ATTO 565 and ATTO 647N at
30 nm fit 99–100 % (0–17 % before); at 40–200 nm the widths, position scatter and fraction fitted
are those of the unpatched fitter. A fit that ends with both widths negative (the model depends on
them only through their product and squares) is given their absolute values instead of being
rejected.

The patch is on the fitter class, so the worker processes the fits run in (forked on Linux)
inherit it. Restart the kernel to undo it.

In [ ]:
from scipy.optimize import leastsq
import pyS3M.gaussoptfuncs as gof
from pyS3M import ImageAnalysisFunctions as IAF

BG_GUESS_FLOOR_PE = 0.1


def _wide_psf_initial_guess(self, smoothed_punctum, punctum, masks):
    ig = gof.initial_guess(smoothed_punctum, punctum, masks).copy()
    size, n_ch = punctum.shape[0], masks.shape[-1]
    gauss = gof.gaussian_unscaled_model(np.zeros((size, size)), np.arange(size, dtype=np.float64), size,
                                        ig[0], ig[1], ig[2], ig[3])
    ring = np.ones((size, size), dtype=bool)
    b = max(2, size // 8)
    ring[b:-b, b:-b] = False
    for c in range(n_ch):
        px = masks[:, :, c].astype(bool)
        bg = max(float(punctum[px & ring].mean()), BG_GUESS_FLOOR_PE)
        ig[4 + c] = np.sqrt(bg)
        ig[4 + n_ch + c] = np.sqrt(max((punctum[px] - bg).sum() / max(gauss[px].sum(), 1e-12), 1.0))
    return ig


def _leastsq_step_abs_widths(self, x0, data, masks, weights, size):
    pfit, pcov, _, _, ok = leastsq(
        gof.WLS_chi_nobounds, x0=x0, args=(data, masks, weights, size, size * size), full_output=True,
        ftol=IAF.FittingConstants.DEFAULT_FTOL, xtol=IAF.FittingConstants.DEFAULT_XTOL,
    )
    if pfit[2] < 0 and pfit[3] < 0:   # same model with both widths positive
        pfit = pfit.copy()
        pfit[2:4] = -pfit[2:4]
        if pcov is not None:
            flip = np.ones(len(pfit))
            flip[2:4] = -1
            pcov = pcov * np.outer(flip, flip)
    return pfit, pcov, ok


IAF.StandardDataFittingProcessor._generate_initial_guess = _wide_psf_initial_guess
IAF.StandardDataFittingProcessor._leastsq_step = _leastsq_step_abs_widths
print("STANDARD_DATA fitter patched for wide PSFs (this kernel only)")

---
## 2 — Simulate and fit

One `test_simulation_method` call per dye and pixel width. Re-running skips points whose results
exist (unless `OVERWRITE`).

In [ ]:
def raw_file(dye, pixel_nm):
    return SIM_DIR / f"{flag(pixel_nm)}LM_method_{dye.replace('/', '-')}_rawresults.h5"


todo = [(d, p) for d in DYES for p in PIXEL_SIZES if OVERWRITE or not raw_file(d, p).exists()]
print(f"{len(todo)} of {len(DYES) * len(PIXEL_SIZES)} points to simulate")
t0 = time.time()
for i, (dye, pixel_nm) in enumerate(todo, 1):
    config = SimulationConfig(
        n_bootstrap=N_BOOTSTRAP, background_photons=0.0, NA=NA, pixel_size=float(pixel_nm),
        save_raw_results=True, subtractx0y0=False, saverawimages=False, verbose=False,
        use_stochastic_photons=True, n_unit_cells=n_unit_cells(pixel_nm),
    )
    MSF.test_simulation_method(
        dye=dye, filters=[], wavelength=wavelength, camera_parameters=camera_params,
        save_folder=str(SIM_DIR), n_photon_space=np.array([N_PHOTONS], dtype=float),
        smoothing_function=smoothing_function, strategy=FittingStrategy.STANDARD_DATA,
        starting_flag=flag(pixel_nm), config=config, overwrite=True,
    )
    elapsed = (time.time() - t0) / 60
    print(f"[{i}/{len(todo)}] {dye} {pixel_nm:.0f} nm  ({elapsed:.1f} min, ETA {elapsed / i * (len(todo) - i):.0f} min)",
          flush=True)

---
## 3 — Precision per dye and pixel width

Fits outside the image or with non-positive widths are dropped (as for the SBR figures); the
fraction of molecules with a kept fit is reported too (the fit gate rejects some at large pixels).

In [ ]:
metrics = ['sigma_xy', 'sigma_colour', 'fraction_fitted']
PX_XR = xr.DataArray(np.full((len(DYES), len(PIXEL_SIZES), len(metrics)), np.nan),
                     coords={'dye': DYES, 'pixel_nm': PIXEL_SIZES, 'metric': metrics},
                     dims=['dye', 'pixel_nm', 'metric'])

for i_d, dye in enumerate(DYES):
    dye_str = dye.replace('/', '-')
    for i_p, pixel_nm in enumerate(PIXEL_SIZES):
        h5 = raw_file(dye, pixel_nm)
        gt = sorted(glob.glob(str(SIM_DIR / f"{flag(pixel_nm)}*{dye_str}*groundtruth*.csv")))
        ip = sorted(glob.glob(str(SIM_DIR / f"{flag(pixel_nm)}*{dye_str}*input_parameters*.csv")))
        if not h5.exists() or not gt or not ip:
            print(f"missing: {dye} {pixel_nm:.0f} nm")
            continue
        truth = pd.read_csv(gt[0])
        truth_px = np.vstack([truth['x0'].to_numpy(), truth['y0'].to_numpy()]).T / pixel_nm
        fingerprint = pd.read_csv(ip[0]).to_numpy()[0][-3:].astype(float)
        fingerprint = fingerprint[None, :] / fingerprint.sum()
        fits = pd.read_hdf(str(h5), key='data')
        fits = fits[fits['photon_level'] == 0]
        side = n_unit_cells(pixel_nm) * mosaic.shape[0]
        ok = ((fits['xc'] > 0) & (fits['yc'] > 0) & (fits['s_x'] > 0) & (fits['s_y'] > 0)
              & (fits['xc'] < side) & (fits['yc'] < side)).to_numpy()
        ex = fits['xc'].to_numpy()[ok] - truth_px[ok, 0]
        ey = fits['yc'].to_numpy()[ok] - truth_px[ok, 1]
        colours = fits[['A_B', 'A_G', 'A_R']].to_numpy()[ok]
        PX_XR[i_d, i_p, 0] = np.sqrt((np.nanstd(ex) ** 2 + np.nanstd(ey) ** 2) / 2) * pixel_nm
        PX_XR[i_d, i_p, 1] = np.nanstd(cdist(colours, fingerprint))
        PX_XR[i_d, i_p, 2] = np.isfinite(ex).sum() / len(truth)

PX_XR.to_netcdf(str(SIM_DIR / 'PixelSize_XR.nc'))
print(PX_XR.sel(metric='fraction_fitted').to_pandas().T.round(3).to_string())

---
## 4 — Figure (`Pixel_Size_StandardUnits`)

In [ ]:
fig, axs = plotter.two_column_plot(ncols=2)
for dye, colour in zip(DYES, COLOURS):
    axs[0].scatter(PIXEL_SIZES, PX_XR.sel(dye=dye, metric='sigma_xy'), s=10, lw=0, color=colour,
                   label=f"{dye}, {N_PHOTONS:,} photons")
    axs[1].scatter(PIXEL_SIZES, PX_XR.sel(dye=dye, metric='sigma_colour'), s=10, lw=0, color=colour)
axs[0].set_ylabel(r'$\sigma_{xy}$/nm')
axs[1].set_ylabel(r'$\sigma_{\mathrm{colour}}$/relative pixel QE')
for ax, label in zip(axs, 'ab'):
    ax.grid(color='gray', lw=0.5, ls='--', alpha=0.5)   # grid lines, as published
    ax.set_axisbelow(True)
    ax.axvline(XIMEA_PIXEL_NM, ls='--', color='red', lw=0.8)
    ax.set_xlabel('pixel width/nm')
    ax.set_xlim(PIXEL_SIZES[0] - 5, PIXEL_SIZES[-1] + 5)
    ax.set_ylim(bottom=0)
    ax.text(-0.12, 1.02, f'{label})', transform=ax.transAxes, fontsize=8, fontweight='bold')
axs[0].legend(fontsize=6, frameon=False)
plotter.save_or_show(fig, save_path=str(FIG_DIR / 'Pixel_Size_StandardUnits.svg'), show=True)